In [1]:
import requests
from pathlib import Path

# 1. Defina a URL do GitHub
urlGit = "https://raw.githubusercontent.com/leogiarola/I2A2/refs/heads/main/Modulo%2002/Atividades/Notas%20Fiscais/notas-fiscais%20(3).csv"

# 2. Faça o download do conteúdo
response = requests.get(urlGit)
response.raise_for_status()  # Verifica se houve erro

# 3. Crie a pasta destino dentro do Colab
target_folder = Path('/content/dados_rag_multiformato')
target_folder.mkdir(parents=True, exist_ok=True)

# 4. Defina o arquivo de destino
file_path = target_folder / 'dados_notas_fiscais.csv'

# 5. Salve o conteúdo baixado
file_path.write_bytes(response.content)

print(f"✅ Download concluído! Arquivo salvo em: {file_path}")


✅ Download concluído! Arquivo salvo em: \content\dados_rag_multiformato\dados_notas_fiscais.csv


In [5]:
# Instalar as bibliotecas que usaremos
!pip install -q pdfplumber langchain langchain_community sentence-transformers faiss-cpu python-pptx pandas langchain-openai langchain-huggingface transformers torch accelerate ragas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7

In [4]:
from google.colab import userdata
OPENAI_KEY = userdata.get('OPENAI_API_KEY')

In [6]:
import pandas as pd
import pdfplumber
from pptx import Presentation

#Funções para leitura de diferentes tipos de arquivo
def read_txt(path):
    #Lê o conteúdo de um arquivo .txt
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        print(f"Erro ao ler o arquivo TXT {path}: {e}")
        return ""

def read_pdf(path):
    #Extrai o texto de um arquivo .pdf
    try:
        with pdfplumber.open(path) as pdf:
            texto = ""
            for pagina in pdf.pages:
                texto += pagina.extract_text() + "\n"
        return texto
    except Exception as e:
        print(f"Erro ao ler o arquivo PDF {path}: {e}")
        return ""

def read_pptx(path):
    #Extrai o texto de uma apresentação .pptx
    try:
        apresentacao = Presentation(path)
        texto = ""
        for slide in apresentacao.slides:
            for shape in slide.shapes:
                if hasattr(shape, "text"):
                    texto += shape.text + "\n"
        return texto
    except Exception as e:
        print(f"Erro ao ler o arquivo PPTX {path}: {e}")
        return ""

def read_csv(path):
    #Lê um arquivo .csv e o converte para uma string formatada
    try:
        df = pd.read_csv(path, sep=';')
        return df.to_string()
    except Exception as e:
        print(f"Erro ao ler o arquivo CSV {path}: {e}")
        return ""

def carregar_documentos_do_diretorio(caminho_diretorio):
    documentos = []
    print(f"Lendo arquivos do diretório: {caminho_diretorio}")
    for nome_arquivo in os.listdir(caminho_diretorio):
        caminho_completo = os.path.join(caminho_diretorio, nome_arquivo)
        conteudo = ""
        if nome_arquivo.endswith(".txt"):
            conteudo = read_txt(caminho_completo)
        elif nome_arquivo.endswith(".pdf"):
            conteudo = read_pdf(caminho_completo)
        elif nome_arquivo.endswith(".pptx"):
            conteudo = read_pptx(caminho_completo)
        elif nome_arquivo.endswith(".csv"):
            conteudo = read_csv(caminho_completo)

        if conteudo:
            documento = Document(
                page_content=conteudo,
                metadata={"fonte": nome_arquivo}
            )
            documentos.append(documento)
            print(f" - Arquivo '{nome_arquivo}' carregado com sucesso.")
    return documentos

In [8]:
from langchain_core.documents import Document
import logging
import os

logging.getLogger("pdfminer").setLevel(logging.ERROR)

#Carrega todos os documentos da pasta
documentos_carregados = carregar_documentos_do_diretorio("dados_rag_multiformato")
print(f"\nTotal de documentos processados: {len(documentos_carregados)}")

Lendo arquivos do diretório: dados_rag_multiformato
 - Arquivo 'dados_notas_fiscais.csv' carregado com sucesso.

Total de documentos processados: 1


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

#Divide os documentos em chunks que serão processados pelo retriever
print("\nDividindo os documentos em chunks...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
documentos_divididos = text_splitter.split_documents(documentos_carregados)
print(f"Total de chunks criados: {len(documentos_divididos)}")

dados_para_df = [
    {
        'fonte': doc.metadata['fonte'],
        'conteudo': doc.page_content,
    }
    for doc in documentos_divididos
]
df = pd.DataFrame(dados_para_df)
display(df)


Dividindo os documentos em chunks...
Total de chunks criados: 16750


,fonte,conteudo
0,notas-fiscais (4).csv,ORGÃO SUPERIOR DESTINATÁRIO ...
1,notas-fiscais (4).csv,FORNECEDOR CNPJ DO FORNECEDOR MUNICÍPIO ...
2,notas-fiscais (4).csv,0 ...
3,notas-fiscais (4).csv,SEARA ALIMENTOS LTDA 02.914.460/0281-60 ...
4,notas-fiscais (4).csv,1 ...
...,...,...
16745,notas-fiscais (4).csv,EDITORA MODERNA LTDA. 62.136.304/0038-20 ...
16746,notas-fiscais (4).csv,8372 ...
16747,notas-fiscais (4).csv,DO BRASIL LTDA 25.210.463/0003-70 ...
16748,notas-fiscais (4).csv,8373 ...


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

#Cria os embeddings e o Vector Store
print("\nGerando embeddings e criando o Vector Store...")
embedding_function = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")
db = FAISS.from_documents(documentos_divididos, embedding_function)
print("Vector Store criado com sucesso.")

#Define o template do prompt para o LLM
template_prompt = """
### Instruções:
Você é um assistente pessoal responsável por responder perguntas dos usuários, com base no contexto fornecido.
Sempre que possível, cite a fonte da sua informação no final da resposta, como por exemplo: (Fonte: nome_do_arquivo).

### Documentos:
{context}

### Pergunta:
{query}

### Resposta:
"""
prompt = PromptTemplate(
    input_variables=["context", "query"],
    template=template_prompt,
)


Gerando embeddings e criando o Vector Store...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Vector Store criado com sucesso.


In [ ]:
#Importando e configurando o LLM
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_openai import ChatOpenAI
import os

os.environ["OPENAI_API_KEY"] = OPENAI_KEY

try:
    llm_gpt = ChatOpenAI(model_name="gpt-4o-mini")
    print("LLM configurado com sucesso.")
except Exception as e:
    print(f"\nErro ao configurar o LLM: {e}")
    llm_gpt = None

LLM configurado com sucesso.


In [ ]:
def formatar_documentos(docs):
    return "\n\n".join(f"Fonte: {doc.metadata['fonte']}\nConteúdo: {doc.page_content}" for doc in docs)


#Configura retriever com o número de chunks a serem recuperados
retriever = db.as_retriever(search_kwargs={'k': 5})

#Cria a chain de execução
rag_gpt = (
    {"context": retriever | formatar_documentos, "query": RunnablePassthrough()}
    | prompt
    | llm_gpt
    | StrOutputParser()
)

In [ ]:
#Testando bot
pergunta = "Liste todos os órgãos destinatários presentes nos dados"
print(f"\n--- Executando a pergunta: '{pergunta}' ---")

#Recuperando o contexto relevante
documentos_contexto = retriever.invoke(pergunta)
print("\n--- Contexto Recuperado para a Pergunta ---")
print(formatar_documentos(documentos_contexto))
print("-------------------------------------------\n")

print("\n>>> Resposta do modelo:")
resultado_gpt = rag_gpt.invoke(pergunta)

print(resultado_gpt)


--- Executando a pergunta: 'Liste todos os órgãos destinatários presentes nos dados' ---

--- Contexto Recuperado para a Pergunta ---
Fonte: notas-fiscais (4).csv
Conteúdo: 6099                                                        Ministério da Educação                                                                      Universidade Federal do Rio de Janeiro      33.663.683/0053-47                                        BAXTER HOSPITALAR LTDA  49.351.786/0011-52

Fonte: notas-fiscais (4).csv
Conteúdo: 6098                                                        Ministério da Educação                                                                      Universidade Federal do Rio de Janeiro      33.663.683/0053-47                                        BAXTER HOSPITALAR LTDA  49.351.786/0011-52

Fonte: notas-fiscais (4).csv
Conteúdo: 6634                                                           Ministério da Saúde                                                                      